# ToyGPT Playground
All library code lives in the `toygpt` package (`src/toygpt/`); install it once with `pip install -e .` from the project root.
This notebook is for poking at things interactively. For full, reproducible runs use `experiments/train_gpt.py --config ...`.
`autoreload` picks up edits to the package without restarting the kernel.

In [1]:
import os, sys
if not os.path.exists("/content/lm-playground"):
    !git clone https://github.com/Sahlar7/lm-playground.git /content/lm-playground
else:
    !git -C /content/lm-playground pull
!pip install -q regex
sys.path.insert(0, "/content/lm-playground/src")

Cloning into '/content/lm-playground'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 46 (delta 7), reused 46 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 44.82 KiB | 376.00 KiB/s, done.
Resolving deltas: 100% (7/7), done.


In [2]:
try:
    %load_ext autoreload
    %autoreload 2
except ModuleNotFoundError:
    print("autoreload unavailable; restart the runtime after editing package code")

import torch
from toygpt.models import GPTConfig, ToyGPT
from toygpt.tokenization import BPETokenizer
from toygpt.data import load_wikitext, build_lm_datasets
from toygpt.training import TrainingConfig, train
from toygpt.inference import generate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

autoreload unavailable; restart the runtime after editing package code
cuda


In [3]:
train_corpus, val_corpus = load_wikitext()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

## Our BPE tokenizer

In [4]:
bpe_tokenizer = BPETokenizer(vocab_size=4000)  # You can adjust vocab_size
bpe_tokenizer.train(train_corpus)
ids = bpe_tokenizer.encode("the cat sat on the mat")
print(ids)
print(bpe_tokenizer.decode(ids))

Pretokenizing...
Creating vocab...
Training...
[116, 257, 1931, 3221, 325, 260, 1145]
the cat sat on the mat


## Train ToyGPT
Using the pretrained GPT-2 tokenizer for now. To use our BPE tokenizer instead, swap in `bpe_tokenizer`
and `GPTConfig(vocab_size=len(bpe_tokenizer.vocab))`.

In [5]:
from transformers import GPT2Tokenizer
pretrained_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

model_config = GPTConfig(vocab_size=pretrained_tokenizer.vocab_size)
model = ToyGPT(model_config)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"The model has {total_params:,} trainable parameters.")

train_dataset, val_dataset = build_lm_datasets(pretrained_tokenizer, train_corpus, val_corpus, model_config.max_seq_len)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

The model has 70,434,816 trainable parameters.


In [6]:
# num_workers=0: DataLoader worker processes are unreliable inside Jupyter on Windows
train_config = TrainingConfig(n_epochs=15, num_workers=0, out_dir="../runs/playground")
checkpoint = train(model, train_dataset, val_dataset, train_config, device)

epoch   0  train loss 7.2948  val loss 6.7025
  saved new best (val loss 6.7025)
epoch   1  train loss 6.4431  val loss 6.3651
  saved new best (val loss 6.3651)
epoch   2  train loss 6.0732  val loss 6.1343
  saved new best (val loss 6.1343)
epoch   3  train loss 5.7901  val loss 5.9832
  saved new best (val loss 5.9832)
epoch   4  train loss 5.5597  val loss 5.8625
  saved new best (val loss 5.8625)
epoch   5  train loss 5.3628  val loss 5.7643
  saved new best (val loss 5.7643)
epoch   6  train loss 5.1918  val loss 5.6911
  saved new best (val loss 5.6911)
epoch   7  train loss 5.0362  val loss 5.6288
  saved new best (val loss 5.6288)
  saved new best (val loss 5.5439)
epoch  10  train loss 4.6327  val loss 5.5103
  saved new best (val loss 5.5103)
epoch  11  train loss 4.5126  val loss 5.4891
  saved new best (val loss 5.4891)
epoch  12  train loss 4.3973  val loss 5.4746
  saved new best (val loss 5.4746)
epoch  13  train loss 4.2846  val loss 5.4648
  saved new best (val loss 5

## Generate

In [7]:
model.load_state_dict(torch.load(checkpoint, map_location=device))
for prompt in ["The history of", "In the United States", "The game began"]:
    print(generate(model, pretrained_tokenizer, prompt, max_new_tokens=50))
    print()

The history of Phil Spector ) , Sieoina , Jack and Sabby , in Los Angeles , is unresolved . These , argued perform clear teachings , which are mined Castle , sign , is known for cleaning and symptoms . The earliest parodies are reborn due to

In the United States , called primarily in the domestic disease . 
 In two weeks later , Magman and as errors , at a scenichouse Cup back to New York to England , in date , contributed to the Springfield . The Third Loyals known in the National

The game began in late February 1918 . He decided for when he was McElroy on May 30 , but scored a 14 goals in a quarter @-@ yard pass " . Hisesca Burowie , Suez Kanye Chronicles II was the first ever played

